# 14｜卷積神經網路_影像特徵與架構

手算與程式同時檢查 2D 互相關、ReLU 與最大池化。

從上到下執行程式格。第一格安裝缺少的套件，第二格準備本章所需設定。

## 投影片與程式的閱讀方式

對應 `slides/14_卷積神經網路_影像特徵與架構.md`，目前 38 頁。逐圖完整表見 `programs/SLIDE_NOTEBOOK_MAP_05_25.md`。

NCHW 卷積、padding/pooling、Inception/Residual/Depthwise/SE 前向與增強。

各格會標示實驗資料、評估方式與適用範圍；架構圖用於說明機制與張量形狀。

| 程式代碼 | 投影片頁次 | 本格學習內容 |
|---|---|---|
| 14-03 | 8、10、11、17 | 課堂小實驗 |
| 14-04 | 3、10、11 | 執行結果圖 |
| 14-05 | 4、5、7、8、9、10、11、12、13、14、15、16、17、18、19、38 | 1. 可重用的 NCHW 卷積與池化 |
| 14-06 | 20、21、22、23、26、27、28、29、30、31、32、33、34、35、36、37、38 | 2. CNN、Inception、Residual、Depthwise 與 SE 的前向 |
| 14-07 | 22、24、25、35、38 | 3. 增強的標籤語意與分類損失 |

In [ ]:
# 第 1 格：安裝本 Notebook 實際使用的套件。
# 程式 14-01｜投影片 14：2 頁
import importlib.util, subprocess, sys
_needed = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.0,<3', 'matplotlib': 'matplotlib>=3.7,<4'}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])

In [ ]:
# 第 2 格：本 Notebook 的輸出位置與繪圖設定。
# 程式 14-02｜投影片 14：2 頁
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
ROOT = Path.cwd()
if ROOT.name == "notebooks" and (ROOT.parent / "pyproject.toml").exists():
    ROOT = ROOT.parent
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    svg_path = folder / f"{name}.svg"
    plt.gcf().savefig(svg_path, bbox_inches="tight", metadata={"Date": None})
    svg_path.write_text("\n".join(line.rstrip() for line in svg_path.read_text().splitlines()) + "\n")
    plt.show()
    plt.close(plt.gcf())

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(
        json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")

## 課堂小實驗

In [ ]:
# 程式 14-03｜投影片 14：8、10、11、17 頁
X = np.array([[1, 2, 0], [0, 1, 3], [2, 1, 0]], dtype=float)
K = np.array([[1, 0], [0, -1]], dtype=float)
Y = np.empty((2, 2))
for i in range(2):
    for j in range(2):
        Y[i, j] = np.sum(X[i:i+2, j:j+2] * K)  # 深度學習框架採互相關慣例
relu = np.maximum(Y, 0)
pooled = relu.max()
print('cross-correlation output:\n', Y)
print('after ReLU:', relu.tolist(), ' max pool:', pooled)

## 執行結果圖

In [ ]:
# 程式 14-04｜投影片 14：3、10、11 頁
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, arr, title in zip(axes, [X, K, Y], ['input', 'kernel', 'output']):
    im = ax.imshow(arr, cmap='coolwarm'); ax.set_title(title)
    for (i, j), value in np.ndenumerate(arr): ax.text(j, i, f'{value:g}', ha='center', va='center')
    ax.set_xticks([]); ax.set_yticks([])
savefig('14_convolution_demo')
report('convolution_demo', {'output': Y.tolist(), 'relu': relu.tolist(), 'max_pool': pooled})

## 1. 可重用的 NCHW 卷積與池化

輸入是 [批次,通道,高,寬]；權重是 [輸出通道,輸入通道,核高,核寬]。函式採互相關、不翻轉核。用合成邊緣展示局部感受野、特徵圖、padding 與池化。

In [ ]:
# 程式 14-05｜投影片 14：4、5、7、8、9、10、11、12、13、14、15、16、17、18、19、38 頁
from numpy.lib.stride_tricks import sliding_window_view
def conv14(x,w,padding=0,stride=1):
    x=np.pad(x,((0,0),(0,0),(padding,padding),(padding,padding)))
    patches=sliding_window_view(x,w.shape[-2:],axis=(-2,-1))[:,:,::stride,::stride]
    return np.einsum('bchwij,ocij->bohw',patches,w)
def pool14(x):
    b,c,h,w=x.shape;assert h%2==0 and w%2==0
    return x.reshape(b,c,h//2,2,w//2,2).max(axis=(3,5))
assert np.allclose(conv14(X[None,None],K[None,None])[0,0],Y)
rng14=np.random.default_rng(SEED);img14=np.zeros((1,1,16,16));img14[:,:,4:12,5:11]=1
kernels14=np.array([[[[-1,0,1],[-1,0,1],[-1,0,1]]],[[[-1,-1,-1],[0,0,0],[1,1,1]]]],float)
features14=conv14(img14,kernels14,padding=1);pooled14=pool14(np.maximum(features14,0))
fig,axes=plt.subplots(1,4,figsize=(12,3))
for ax,a,title in zip(axes,[img14[0,0],features14[0,0],features14[0,1],pooled14[0,0]],['input','vertical filter','horizontal filter','ReLU + pooling']):ax.imshow(a,cmap='coolwarm');ax.set_title(title)
from matplotlib.patches import Rectangle
axes[0].add_patch(Rectangle((4.5,3.5),3,3,fill=False,edgecolor='red',lw=2))
savefig('c14_feature_maps')
fig,axes=plt.subplots(1,2,figsize=(7,3))
for ax,p in zip(axes,[0,1]):
    out14=conv14(img14,kernels14,padding=p);ax.imshow(out14[0,0]);ax.set_title(f'padding={p}, shape={out14.shape[-2:]}')
savefig('c14_padding')
x14=rng14.normal(size=(2,3,8,8));w14=rng14.normal(size=(4,3,3,3));o14=conv14(x14,w14,padding=1,stride=2)
assert o14.shape==(2,4,4,4)
contributions14=[conv14(x14[:,j:j+1],w14[:,j:j+1],padding=1,stride=2) for j in range(3)]
assert np.allclose(sum(contributions14),o14)
print('Multichannel sum shape / parameters:',o14.shape,w14.size+4)
# 一格平移不保證池化不變
shift14=np.roll(img14,1,axis=-1)
print('One-pixel shift changes pooled output:',not np.array_equal(pool14(img14),pool14(shift14)))

## 2. CNN、Inception、Residual、Depthwise 與 SE 的前向

以下是縮小的構件與形狀示範，權重未訓練，不代表已重現 LeNet、AlexNet、ResNet 或 MobileNet 的辨識率。每個相加／串接都有實際張量檢查。

In [ ]:
# 程式 14-06｜投影片 14：20、21、22、23、26、27、28、29、30、31、32、33、34、35、36、37、38 頁
batch14=rng14.normal(size=(4,1,28,28))
h14=np.maximum(conv14(batch14,rng14.normal(0,.1,(4,1,3,3)),padding=1),0)
h14=pool14(h14);h14=np.maximum(conv14(h14,rng14.normal(0,.1,(8,4,3,3)),padding=1),0)
gap14=h14.mean(axis=(2,3));logits14=gap14@rng14.normal(size=(8,10))
assert logits14.shape==(4,10);print('CNN shapes:',batch14.shape,h14.shape,gap14.shape,logits14.shape)
# 同尺寸平行分支串接 channel
branch1=conv14(x14,rng14.normal(size=(2,3,1,1)))
branch3=conv14(x14,rng14.normal(size=(2,3,3,3)),padding=1)
branch5=conv14(x14,rng14.normal(size=(2,3,5,5)),padding=2)
inception14=np.concatenate([branch1,branch3,branch5],axis=1);assert inception14.shape==(2,6,8,8)
# stride=2 或 channel 改變，都需要 projection
main14=conv14(x14,rng14.normal(size=(5,3,3,3)),padding=1,stride=2)
skip14=conv14(x14,rng14.normal(size=(5,3,1,1)),stride=2)
res14=np.maximum(main14+skip14,0);assert res14.shape==(2,5,4,4)
# Depthwise 每通道各算，再 pointwise 混合
wd14=rng14.normal(size=(3,1,3,3))
depth14=np.concatenate([conv14(x14[:,j:j+1],wd14[j:j+1],padding=1) for j in range(3)],axis=1)
point14=conv14(depth14,rng14.normal(size=(5,3,1,1)));assert point14.shape==(2,5,8,8)
# Squeeze -> excitation -> channel reweight
squeeze14=x14.mean(axis=(2,3));gate14=1/(1+np.exp(-np.maximum(squeeze14@rng14.normal(size=(3,2)),0)@rng14.normal(size=(2,3))))
se14=x14*gate14[:,:,None,None];assert se14.shape==x14.shape
fig,axes=plt.subplots(1,2,figsize=(9,3))
axes[0].bar(['input','Inception','residual','depth+point'],[3,6,5,5]);axes[0].set_ylabel('output channels')
axes[1].bar(np.arange(3),gate14[0]);axes[1].set(xlabel='channel',ylabel='SE gate');savefig('c14_architecture_channels')
assert (9*256*128,256*32+9*32*128)==(294912,45056)
assert (9*32*64,9*32+32*64)==(18432,2336)
print('Bottleneck MACs:',294912,45056,'ordinary/depthwise parameters:',18432,2336)

## 3. 增強的標籤語意與分類損失

合成物件的水平反轉保留「有物件」標籤；數字、文字、醫療方向標籤不一定可反轉。模型的 logits 先算穩定 log-softmax，再取正確類別的負對數。

In [ ]:
# 程式 14-07｜投影片 14：22、24、25、35、38 頁
views14=[img14[0,0],np.fliplr(img14[0,0]),np.clip(img14[0,0]*.6+.2,0,1),np.roll(img14[0,0],2,axis=0)]
fig,axes=plt.subplots(1,4,figsize=(10,3))
for ax,a,title in zip(axes,views14,['original','horizontal flip','brightness','translation']):ax.imshow(a,cmap='gray',vmin=0,vmax=1);ax.set_title(title);ax.axis('off')
savefig('c14_augmentation')
shifted14=logits14-logits14.max(1,keepdims=True);logp14=shifted14-np.log(np.exp(shifted14).sum(1,keepdims=True))
y14=np.array([0,1,2,3]);ce14=-logp14[np.arange(4),y14].mean();print('Untrained forward CE:',ce14)
assert np.isfinite(ce14)

## 練習：改動一個參數

先預測結果，再修改程式中的一個參數並重跑；比較圖表或數值，說明差異。